# TESS SPOC Availability — Batch 2 (HJ & UHJ)
Queries TESS SPOC 2-min cadence for the 5 new Hot Jupiters and 5 new Ultra-Hot Jupiters in batch 2. Merges results with existing batch 1 `tess_availability.csv`.

In [1]:
import os
import pandas as pd
import lightkurve as lk
from IPython.display import display

BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), '..'))
FOLDER   = os.path.join(BASE_DIR, 'Final HJ and UHJ')

hj_b2  = pd.read_csv(os.path.join(FOLDER, 'hot_jupiters_batch2.csv'))
uhj_b2 = pd.read_csv(os.path.join(FOLDER, 'ultra_hot_jupiters_batch2.csv'))

print('Hot Jupiters — Batch 2:')
display(hj_b2[['Planet', 'Transit_depth_pct', 'Period_days', 'Radius_Rjup']])
print()
print('Ultra-Hot Jupiters — Batch 2:')
display(uhj_b2[['Planet', 'Transit_depth_pct', 'Period_days', 'Radius_Rjup']])

Hot Jupiters — Batch 2:


c:\Users\pnayg\Desktop\cvif-astro-p1\.venv\Lib\site-packages\lightkurve\prf\__init__.py:7: UserWarning: Warning: the tpfmodel submodule is not available without oktopus installed, which requires a current version of autograd. See #1452 for details.
  warnings.warn(


,Planet,Transit_depth_pct,Period_days,Radius_Rjup
0,CoRoT-02,2.772945,1.743,1.46
1,Kepler-447,2.710057,7.794,1.65
2,NGTS-03,2.674500,1.675,1.48
3,TrES-3,2.672394,1.306,1.31
4,TOI-1855,2.653087,1.364,1.65



Ultra-Hot Jupiters — Batch 2:


,Planet,Transit_depth_pct,Period_days,Radius_Rjup
0,OGLE-TR-056,1.052407,1.212,1.7340
1,WASP-076,1.001112,1.810,1.7730
2,TOI-1518,0.976379,1.903,1.8750
3,WASP-018,0.952128,0.941,1.1926
4,GPX-1,0.937714,1.745,1.4700


In [2]:
def query_planet(planet_name):
    try:
        sr = lk.search_lightcurve(
            planet_name, mission='TESS', cadence=120, author='SPOC'
        )
        if len(sr) == 0:
            return None, 'No TESS SPOC 2-min data found'
        tbl = sr.table.to_pandas()
        if 'sequence_number' in tbl.columns:
            tbl = tbl.rename(columns={'sequence_number': 'sector'})
        elif '#obs_id' in tbl.columns:
            tbl['sector'] = tbl['#obs_id'].str.extract(r's(\d+)', expand=False)
        for col in ['t_min', 't_max']:
            if col in tbl.columns:
                tbl[col] = tbl[col].round(2)
        out_cols = [c for c in ['target_name', 'sector', 't_min', 't_max', 'exptime'] if c in tbl.columns]
        return tbl[out_cols], f'{len(tbl)} sector(s) found'
    except Exception as e:
        return None, f'ERROR: {e}'

## Hot Jupiters — Batch 2

In [3]:
hj_summary = []

for _, row in hj_b2.iterrows():
    planet = row['Planet']
    depth  = row['Transit_depth_pct']
    print(f'Querying: {planet}  (depth={depth:.3f}%)')

    tbl, msg = query_planet(planet)
    print(f'  -> {msg}')

    if tbl is not None:
        display(tbl)
        sectors = sorted(tbl['sector'].dropna().astype(str).tolist()) if 'sector' in tbl.columns else []
        hj_summary.append({
            'Planet': planet, 'Type': 'HJ', 'Batch': 'HJ-B2',
            'Transit_depth_pct': depth, 'N_sectors': len(tbl),
            'Sectors': ', '.join(sectors), 'Status': 'Available'
        })
    else:
        hj_summary.append({
            'Planet': planet, 'Type': 'HJ', 'Batch': 'HJ-B2',
            'Transit_depth_pct': depth, 'N_sectors': 0,
            'Sectors': '', 'Status': msg
        })
    print()

Querying: CoRoT-02  (depth=2.773%)
  -> 2 sector(s) found


,target_name,sector,t_min,t_max,exptime
0,391958006,54,59769.40,59795.63,120.0
1,391958006,81,60506.06,60532.68,120.0



Querying: Kepler-447  (depth=2.710%)
  -> 10 sector(s) found


,target_name,sector,t_min,t_max,exptime
0,48506505,40,59390.15,59418.36,120.0
1,48506505,41,59419.49,59446.08,120.0
2,48506505,53,59743.50,59768.48,120.0
3,48506505,54,59769.40,59795.63,120.0
4,48506505,55,59796.60,59823.76,120.0
5,48506505,74,60312.36,60339.07,120.0
6,48506505,75,60339.28,60366.98,120.0
7,48506505,80,60479.39,60505.84,120.0
8,48506505,81,60506.05,60532.68,120.0
9,48506505,82,60532.89,60558.75,120.0



Querying: NGTS-03  (depth=2.674%)
  -> 3 sector(s) found


,target_name,sector,t_min,t_max,exptime
0,124778445,33,59201.23,59227.07,120.0
1,124778445,87,60662.54,60689.44,120.0
2,124778445,98,60988.00,61045.40,120.0



Querying: TrES-3  (depth=2.672%)
  -> 7 sector(s) found


,target_name,sector,t_min,t_max,exptime
0,116264089,25,58983.13,59008.81,120.0
1,116264089,26,59009.77,59034.64,120.0
2,116264089,40,59390.15,59418.36,120.0
3,116264089,52,59718.14,59742.58,120.0
4,116264089,53,59743.50,59768.48,120.0
5,116264089,79,60452.04,60479.18,120.0
6,116264089,80,60479.39,60505.84,120.0



Querying: TOI-1855  (depth=2.653%)
  -> 1 sector(s) found


,target_name,sector,t_min,t_max,exptime
0,81247740,50,59664.78,59691.01,120.0


## Ultra-Hot Jupiters — Batch 2

In [4]:
uhj_summary = []

for _, row in uhj_b2.iterrows():
    planet = row['Planet']
    depth  = row['Transit_depth_pct']
    print(f'Querying: {planet}  (depth={depth:.3f}%)')

    tbl, msg = query_planet(planet)
    print(f'  -> {msg}')

    if tbl is not None:
        display(tbl)
        sectors = sorted(tbl['sector'].dropna().astype(str).tolist()) if 'sector' in tbl.columns else []
        uhj_summary.append({
            'Planet': planet, 'Type': 'UHJ', 'Batch': 'UHJ-B2',
            'Transit_depth_pct': depth, 'N_sectors': len(tbl),
            'Sectors': ', '.join(sectors), 'Status': 'Available'
        })
    else:
        uhj_summary.append({
            'Planet': planet, 'Type': 'UHJ', 'Batch': 'UHJ-B2',
            'Transit_depth_pct': depth, 'N_sectors': 0,
            'Sectors': '', 'Status': msg
        })
    print()

Querying: OGLE-TR-056  (depth=1.052%)
  -> 4 sector(s) found


,target_name,sector,t_min,t_max,exptime
0,1425441230,39,59361.28,59389.21,120.0
1,1425441246,39,59361.28,59389.21,120.0
2,1425441246,91,60774.79,60802.26,120.0
3,1425441246,92,60802.47,60829.15,120.0



Querying: WASP-076  (depth=1.001%)
  -> 4 sector(s) found


,target_name,sector,t_min,t_max,exptime
0,293435336,30,59115.39,59142.00,120.0
1,293435336,42,59447.19,59472.40,120.0
2,293435336,43,59473.67,59498.33,120.0
3,293435336,97,60933.16,60987.79,120.0



Querying: TOI-1518  (depth=0.976%)
  -> 5 sector(s) found


,target_name,sector,t_min,t_max,exptime
0,427761355,57,59852.86,59881.62,120.0
1,427761355,58,59881.83,59909.55,120.0
2,427761355,77,60394.98,60423.05,120.0
3,427761355,84,60584.09,60609.84,120.0
4,427761355,85,60610.06,60635.55,120.0



Querying: WASP-018  (depth=0.952%)
  -> 6 sector(s) found


,target_name,sector,t_min,t_max,exptime
0,100100827,2,58353.61,58381.02,120.0
1,100100827,3,58381.54,58408.89,120.0
2,100100827,29,59087.74,59113.94,120.0
3,100100827,30,59115.39,59142.49,120.0
4,100100827,69,60181.86,60207.64,120.0
5,100100827,96,60908.03,60932.94,120.0



Querying: GPX-1  (depth=0.938%)
  -> 2 sector(s) found


,target_name,sector,t_min,t_max,exptime
0,245392284,58,59881.83,59909.55,120.0
1,245392284,85,60610.06,60635.55,120.0


## Summary — Merge with Batch 1 and Save

In [5]:
b2_df = pd.DataFrame(hj_summary + uhj_summary)
b2_df = b2_df.sort_values(['Type', 'Transit_depth_pct'], ascending=[True, False])

print('=== TESS SPOC 2-min Availability — Batch 2 ===')
display(b2_df)

out_path = os.path.join(FOLDER, 'tess_availability_batch2.csv')
b2_df.to_csv(out_path, index=False)
print(f'\nSaved: {out_path}')

=== TESS SPOC 2-min Availability — Batch 2 ===


,Planet,Type,Batch,Transit_depth_pct,N_sectors,Sectors,Status
0,CoRoT-02,HJ,HJ-B2,2.772945,2,"54, 81",Available
1,Kepler-447,HJ,HJ-B2,2.710057,10,"40, 41, 53, 54, 55, 74, 75, 80, 81, 82",Available
2,NGTS-03,HJ,HJ-B2,2.674500,3,"33, 87, 98",Available
3,TrES-3,HJ,HJ-B2,2.672394,7,"25, 26, 40, 52, 53, 79, 80",Available
4,TOI-1855,HJ,HJ-B2,2.653087,1,50,Available
5,OGLE-TR-056,UHJ,UHJ-B2,1.052407,4,"39, 39, 91, 92",Available
6,WASP-076,UHJ,UHJ-B2,1.001112,4,"30, 42, 43, 97",Available
7,TOI-1518,UHJ,UHJ-B2,0.976379,5,"57, 58, 77, 84, 85",Available
8,WASP-018,UHJ,UHJ-B2,0.952128,6,"2, 29, 3, 30, 69, 96",Available
9,GPX-1,UHJ,UHJ-B2,0.937714,2,"58, 85",Available



Saved: c:\Users\pnayg\Desktop\cvif-astro-p1\Final HJ and UHJ\tess_availability_batch2.csv
